# RAG Complete Module : LangChain 2026
### Full Pipeline: LLM → Prompt → Output Parser → Loader → Splitter → Embeddings → VectorStore → Retriever → RAG

---

| # | Section |
|---|-----------------------------|
| 1 | Groq LLM Setup              |
| 2 | Prompt Template (LCEL)      |
| 3 | Output Parsers              |
| 4 | Document Loaders            |
| 5 | Text Splitters              |
| 6 | Embeddings                  |
| 7 | Vector Store (FAISS)        |
| 8 | Retriever                   |
| 9 | Full RAG Chain (LCEL)       |

---


## API

- We often need to use **API keys** (like Groq API Key) to connect with external services.  
- Instead of writing the key directly in the code, we store it safely in a hidden file called `.env`.  
- The `dotenv` library helps us **load values from the `.env` file** into our program.  
- `os.getenv("GROQ_API_KEY")` reads the key from environment variables.  
- This way, the code stays clean, secure, and reusable without exposing sensitive information.  
- The print statement checks if the key was loaded successfully:
  - Key found → ready to use  
  - Key missing → you need to set it in `.env` or directly in the code


In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

# For classroom demo — set directly here
# os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxx"

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("✅ Groq API Key Loaded" if GROQ_API_KEY else "❌ Set GROQ_API_KEY")

✅ Groq API Key Loaded


## Required Installations

Before running the code, make sure you install these packages:

- **langchain** → Core framework for building LLM applications  
- **langchain-groq** → Integration with Groq models  
- **python-dotenv** → To load environment variables from `.env` file  
- **groq** → Official Groq Python client (needed for API calls)

### Install with pip:
```bash
pip install langchain langchain-groq python-dotenv groq

## Setting up the LLM (Large Language Model)

- We are using **ChatGroq** to connect with Groq’s hosted LLMs.  
- The `model` parameter chooses which Groq model to run (here: `"llama-3.3-70b-versatile"`).  
- `temperature` controls creativity:
  - Lower (0.0–0.3) → more factual, deterministic answers  
  - Higher (0.7–1.0) → more creative, varied answers  
- `max_tokens` sets the maximum length of the response (here: 1024 tokens).  
- `api_key` is loaded from environment variables (`GROQ_API_KEY`) to authenticate with Groq.  
- Together, this configuration tells the program **which model to use, how creative it should be, how long responses can be, and ensures secure access with the API key**.


## Type of Prompt Used

- We are using a **Chat Prompt Template** in LangChain.  
- This template is made of **two roles**:
  - **System message** → sets the overall behavior of the model.  
    - Example: `"You are a cricket analyst. Answer based on cricket match data."`  
    - This ensures the model always acts like a cricket expert.  
  - **Human message** → represents the user’s input.  
    - Example: `"{question}"` → a placeholder that will be replaced with the actual query.  
- Together, this forms a **Simple Role-based Prompt**:
  - Fixed system role (cricket analyst).  
  - Dynamic human role (student’s question).  
- This type of prompt is best for **classroom demos** because:
  - It is **modular** (easy to reuse).  
  - It is **clear** (students see the role separation).  
  - It is **controlled** (system role keeps answers focused on cricket).


## Types of Prompts Used

### 1. Simple Prompt
- **Structure**: One system message + one human message.
- **System role**: `"You are a cricket analyst. Answer based on cricket match data."`
  - Fixes the model’s behavior as a cricket expert.
- **Human role**: `"{question}"`
  - Placeholder for the student’s query.
- **Use case**: Best for direct Q&A where the model relies on its own knowledge.

---

### 2. RAG Prompt
- **Structure**: System message includes **context injection** + human message.
- **System role**: `"You are a cricket analyst assistant. Use ONLY the following context..."`
  - Ensures the model answers strictly from retrieved context.
  - If answer not found → `"I do not have enough data."`
- **Human role**: `"{question}"`
  - Placeholder for the student’s query.
- **Use case**: Best for **RAG (Retrieval-Augmented Generation)** pipelines where external documents provide the knowledge base.

---

### Key Difference
- **Simple Prompt** → Model answers based on its training + general knowledge.  
- **RAG Prompt** → Model answers only from retrieved context chunks, ensuring accuracy and grounding.  

This shows students how prompt design changes the **source of truth**:
- Simple Prompt → model’s memory.  
- RAG Prompt → external context via retriever.


## Output Parsers in LangChain

- **Output parsers** control how the raw text from the LLM is processed and returned.  
- In this example, we are using two parsers:

### 1. StrOutputParser
- Converts the LLM output into a plain string.  
- Best for free‑form text answers (summaries, explanations, creative writing).  
- No schema or validation — it simply unwraps the text.

### 2. JsonOutputParser
- Converts the LLM output into a JSON object (Python dictionary).  
- Best for structured data (tables, key‑value pairs, records).  
- Ensures the response is valid JSON so it can be programmatically used.

---

## Other Common Parsers

- **MarkdownListOutputParser**  
  - Parses bullet lists into Python lists.  
  - Useful for brainstorming ideas or itemized outputs.

- **PydanticOutputParser**  
  - Validates JSON against a strict schema defined with Pydantic models.  
  - Ensures type safety and catches errors early.  
  - Best for production apps where reliability matters.

- **Structured Output (`with_structured_output`)**  
  - Modern shortcut to enforce schema directly in the LLM call.  
  - Cleaner than manually parsing JSON.  
  - Best for APIs and enterprise workflows.

---

### Key Takeaway
- Use **StrOutputParser** → when plain text is enough.  
- Use **JsonOutputParser** → when structured data is required.  
- Use **MarkdownListOutputParser** → when expecting lists.  
- Use **PydanticOutputParser / Structured Output** → when schema validation is critical.

The print statement confirms that the **LLM, prompts, and parsers** have been successfully reloaded.


In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── LLM ──
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=1024,
    api_key=GROQ_API_KEY
)

# ── Simple Prompt ──
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a cricket analyst. Answer based on cricket match data."),
    ("human", "{question}")
])

# ── RAG Prompt — {context} filled by Retriever, {question} from user ──
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a cricket analyst assistant.
Use ONLY the following context to answer the question.
If the answer is not in the context, say 'I do not have enough data.'

Context:
{context}"""),
    ("human", "{question}")
])

str_parser  = StrOutputParser()
json_parser = JsonOutputParser()

print("LLM + Prompts + Parsers — reloaded")

LLM + Prompts + Parsers — reloaded


## Document Loaders in LangChain

- **Purpose**: Document Loaders are the entry point of a RAG pipeline.  
  They take raw data (files, web pages, JSON, CSV, etc.) and convert them into a standard format called a **Document object**.  
  Each Document has:
  - `page_content` → the actual text content
  - `metadata` → extra info like file name, row index, source URL

---

### Types of Loaders Used Here

1. **TextLoader**
   - Loads plain `.txt` files.
   - Example: `match_summary.txt` → becomes one Document with the entire text.

2. **CSVLoader**
   - Loads `.csv` files row by row.
   - Example: `batting_scorecard.csv` → each row (player stats) becomes a separate Document.
   - Example: `bowling_figures.csv` → each row (bowler stats) becomes a separate Document.

3. **JSONLoader**
   - Loads `.json` files.
   - Can use `jq_schema` to select specific parts of the JSON.
   - Example: `input.json` → each JSON object becomes a Document.

4. **WebBaseLoader**
   - Loads content directly from a webpage.
   - Uses `BeautifulSoup` filters (`bs4.SoupStrainer`) to extract only relevant sections (like tables or main content).
   - Example: IPL points table from CricTracker → scraped into Documents.

---

### Why Document Loaders Matter
- They **normalize different data sources** into a single format (Document objects).  
- This makes downstream steps (splitting, embeddings, vector storage) consistent.  
- Without loaders, every file type would need custom handling.

---

### Analogy
- Think of Document Loaders as **data translators**:
  - TextLoader → translates `.txt` into Document language.
  - CSVLoader → translates rows into Document language.
  - JSONLoader → translates structured JSON into Document language.
  - WebBaseLoader → translates HTML webpages into Document language.
- All translators produce the same output format → Documents.

---

### Key Takeaway
- **Document Loaders = First step of RAG pipeline.**  
- They unify diverse sources (text, CSV, JSON, web) into a common structure.  
- Once loaded, all documents can be merged (`all_docs`) and passed into splitters, embeddings, and vector stores for retrieval.


In [15]:
from langchain_community.document_loaders import TextLoader, WebBaseLoader, JSONLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
import bs4

txt_docs = TextLoader(
    r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt",
    encoding="utf-8"
).load()

batting_docs = CSVLoader(
    r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\batting_scorecard.csv",
    encoding="utf-8"
).load()

bowling_docs = CSVLoader(
    r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\bowling_figures.csv",
    encoding="utf-8"
).load()

json_docs = JSONLoader(
    file_path=r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\input.json",
    jq_schema=".",
    text_content=False
).load()

web_docs = WebBaseLoader(
    web_paths=["https://www.crictracker.com/t20/ipl-indian-premier-league/points-table/?ref=hm"],
    bs_kwargs={"parse_only": bs4.SoupStrainer(
        class_=["entry-content", "standings", "points-table", "table", "main-content"]
    )},
    header_template={"User-Agent": "Mozilla/5.0"}
).load()

all_docs = txt_docs + batting_docs + bowling_docs + json_docs + web_docs

print(f"Documents reloaded : {len(all_docs)} total")
print(f"   TXT={len(txt_docs)} | Batting={len(batting_docs)} | Bowling={len(bowling_docs)} | JSON={len(json_docs)} | Web={len(web_docs)}")

Documents reloaded : 27 total
   TXT=1 | Batting=13 | Bowling=11 | JSON=1 | Web=1


---
### ════════════════════════════════
## TEXT SPLITTERS
### ════════════════════════════════

### Theory : Why Do We Split Documents?

After loading documents we have raw text blobs. Two problems appear:

**Problem 1 : LLM Token Limit**  
LLMs like LLaMA 3.3 have a max context window (8K–128K tokens).  
We cannot dump your entire CSV + TXT + Web page into the LLM at once.

**Problem 2 : Embedding Blur**  
When we embed a 5-page document as one vector, that vector tries to represent  
EVERYTHING at once → it becomes a blur → search accuracy drops badly.

**Solution : Text Splitter**  
Break each document into small, overlapping chunks:
- Small enough to fit in embedding models
- Focused enough for accurate semantic search
- Overlapping so no sentence is cut mid-thought

---
## Key Parameters

| Parameter | What it controls | Typical value |
|---|---|---|
| `chunk_size` | Max characters per chunk | 500 |
| `chunk_overlap` | Characters shared between adjacent chunks | 100 |
| `length_function` | How to measure, chars or tokens | `len` |
| `add_start_index` | Adds chunk position to metadata | `True` |

---
## How Overlap Prevents Lost Context

```
Original text: "Rohit hit a six. The ball went over long-on. The crowd went wild."

WITHOUT overlap:
   Chunk 1: "Rohit hit a six. The ball went over"
   Chunk 2: "long-on. The crowd went wild."
   Chunk 2 loses context, who did what?

WITH overlap (overlap=10):
   Chunk 1: "Rohit hit a six. The ball went over"
   Chunk 2: "went over long-on. The crowd went wild."
   Chunk 2 retains enough context to be understood
```

### RecursiveCharacterTextSplitter *(Best Choice for RAG)*

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
final_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True
)

all_chunks = final_splitter.split_documents(all_docs)

print(f"all_chunks = {len(all_chunks)} chunks")
print(f"These are the INPUT for Section 6 Embeddings")

all_chunks = 46 chunks
These are the INPUT for Section 6 Embeddings


---
# ════════════════════════════════
# EMBEDDINGS
# ════════════════════════════════

## Theory : What Are Embeddings?

**The Core Problem:**  
Computers understand numbers, not words.  
We need to convert text into numbers, but in a smart way that **preserves meaning**.

**Embeddings = a list of numbers (a vector) that represents the meaning of text.**

```
"Rohit hit a century"    → [0.12, 0.85, 0.33, 0.67, ...]  ← 384 numbers
"Sharma scored 100 runs" → [0.11, 0.84, 0.35, 0.65, ...]  ← similar! (same meaning)
"Rain stopped the match" → [0.91, 0.02, 0.78, 0.11, ...]  ← very different
```

---
##  Why Embeddings Are the Heart of RAG

| Without Embeddings (keyword search) | With Embeddings (semantic search) |
|---|---|
| "Rohit" must exactly match "Rohit" | "Rohit" matches "the opener", "the batsman" |
| Synonyms break search | Synonyms work perfectly |
| Context is lost | Context and intent are captured |

---
## We Use: HuggingFace `all-MiniLM-L6-v2`

| Property | Value |
|---|---|
| Cost | **Free** : runs locally |
| Vector size | 384 dimensions |
| Speed | Fast on CPU |
| Quality | Excellent for RAG use cases |

*(Alternative: OpenAI `text-embedding-3-small`, paid, higher accuracy)*

### Embedding Models Overview
#### 10 Popular Models: Learning vs. Industry Use (with Dimension Size)

---

| # | Model Name | Dimensions | Type | Best Use Case | Level |
|---|-------------|------------|------|---------------|--------|
| 1 | **all-MiniLM-L6-v2** | 384 | Sentence Transformer | Fast, local semantic search |  Learning |
| 2 | **paraphrase-MiniLM-L12-v2** | 384 | Sentence Transformer | Text similarity, paraphrase detection |  Learning |
| 3 | **distiluse-base-multilingual-cased-v2** | 512 | Sentence Transformer | Multilingual embeddings |  Learning |
| 4 | **bge-small-en-v1.5** | 512 | BAAI General Embedding | Lightweight, high-quality English embeddings |  Learning |
| 5 | **e5-small-v2** | 384 | Microsoft E5 Series | Retrieval tasks, RAG demos |  Learning |
| 6 | **text-embedding-3-small** | 1,536 | OpenAI | High accuracy, paid API |  Industry |
| 7 | **text-embedding-3-large** | 3,072 | OpenAI | Enterprise-grade semantic search | Industry |
| 8 | **bge-large-en-v1.5** | 1,024 | BAAI General Embedding | Production-level retrieval | Industry |
| 9 | **voyage-large-2** | 1,024 | Voyage AI | Commercial-grade multilingual embeddings | Industry |
| 10 | **cohere-embed-v3** | 4,096 | Cohere | Industry-scale semantic search and classification | Industry |

---

### Summary
- **Learning Models** → Free, lightweight, CPU-friendly, ideal for classroom demos and local RAG setups.  
- **Industry Models** → Paid or large-scale, optimized for production workloads, multilingual support, and enterprise retrieval.

###
Think of **Learning Models** as *bicycles* —> simple, fast, and great for practice.  
**Industry Models** are *sports cars* —> powerful, optimized, and built for large-scale deployment.

---

### Recommended Setup
Use **HuggingFace `all-MiniLM-L6-v2`** for embeddings in local RAG pipeline.  
Later, compare results with **OpenAI `text-embedding-3-small`** to show how accuracy improves with scale.


## Understanding Embedding Dimensions

---

### 1. What Happens When Dimension Size is Bigger (e.g., 3,072 or 4,096)

- **Higher capacity for meaning**  
  - More dimensions = more "slots" to capture subtle relationships in language.  
  - Example: A 384‑dim vector captures broad meaning; a 3,072‑dim vector can encode finer nuances (tone, context, domain‑specific terms).

- **Better accuracy in semantic search**  
  - Larger vectors reduce overlap between unrelated concepts.  
  - Useful in **industry pipelines** where precision matters (legal documents, medical records, enterprise search).

- **Trade‑offs**  
  - **Memory cost**: Larger vectors consume more RAM/disk in vector stores.  
  - **Compute cost**: Similarity search (dot product, cosine similarity) is slower with bigger vectors.  
  - **Best practice**: Use small models (384–512 dims) for learning/demo; use large models (1,536–4,096 dims) for production where accuracy outweighs cost.

---


## 2. How Dimensions Are Created

#### How Embedding Dimensions Are Created: Deep Dive

---

## Step 1: Chunking the Text
- Input text is first **split into chunks** (typically 200–500 tokens).  
- Each chunk is treated as a **semantic unit** that can be independently understood.  
- Example:  
  - Chunk 1 → "Rohit hit a century in Mumbai."  
  - Chunk 2 → "Sharma led India to victory."  


## Step 2: Neural Encoding (Inside the Transformer)
- The embedding model uses **Transformer layers** to process each chunk.  
- **Tokenization** → Words/subwords are converted into numeric IDs.  
- **Positional Encoding** → Adds order information so the model knows sequence.  
- **Self-Attention** → Each token attends to all others (left + right context).  
  - Example: "century" + "Rohit" → cricket meaning.  
  - "century" + "year" → historical meaning.  
- **Hidden States** → Multiple layers refine contextual meaning until the chunk is fully represented.



## Step 3: Vector Representation
- The final output is a **fixed-length vector** (e.g., 384, 1,536, or 4,096 numbers).  
- Each number (dimension) encodes a specific aspect of meaning:  
  - **Word semantics** → actual meaning of terms.  
  - **Syntax** → grammar and sentence structure.  
  - **Context** → relationships between entities (players, matches, places).  
  - **Domain knowledge** → specialized understanding (sports, medicine, finance).  

Example:  

`"Rohit hit a century" → [0.12, 0.85, 0.33, 0.67, ...] → 384 dimensions`



## Step 4: Why Bigger Dimensions Matter
- **Small embeddings (384–512 dims)** → lightweight, fast, good for demos and local RAG.  
- **Large embeddings (1,536–4,096 dims)** → capture subtle nuances, higher accuracy, ideal for industry pipelines.  
- **Trade-offs**:  
  - Larger vectors = more memory + slower search.  
  - But they provide **finer detail**, reducing semantic overlap between unrelated concepts.  



- Dimensions are like **pixels in an image**:  
  - 384 dims = low-resolution photo (broad meaning).  
  - 3,072 dims = high-resolution photo (fine detail).  
- More pixels (dimensions) → clearer picture of meaning, but heavier to store and process.



Embedding dimensions are created by **chunking text → transformer encoding → vector output**.  
Smaller vectors are efficient for practice, while larger vectors are powerful for production where **accuracy and nuance** matter most.

---

### Analogy 

- Think of **dimensions** like **pixels in an image**:  
  - A 384‑dim embedding = a small photo (less detail).  
  - A 3,072‑dim embedding = a high‑resolution photo (fine detail).  
- More pixels (dimensions) → clearer picture of meaning.  
- But bigger images → heavier to store and slower to process.

---

## Key Takeaway

- **Small embeddings (384–512 dims)** → great for demos, fast, lightweight.  
- **Large embeddings (1,536–4,096 dims)** → industry‑grade, capture subtle meaning, better for mission‑critical RAG pipelines.  
- Dimensions are created by analyzing **chunks of text with context windows**, encoding relationships between words into numbers that preserve meaning.


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

# Downloads ~90MB on first run — then cached locally
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}   # normalise for accurate cosine similarity
)

print("Embedding model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3221.62it/s]


Embedding model loaded


### Cosine Similarity 

---

#### 1. Core Idea
Cosine similarity is a mathematical measure that tells us how **similar two vectors are based on their direction**.  
It is widely used in **embeddings** because embeddings represent meaning as vectors in high-dimensional space.  
Two sentences with similar meaning will have vectors pointing in nearly the same direction, even if their lengths differ.



#### 2. Formula


\[
\text{cosine similarity} = \frac{v_1 \cdot v_2}{\|v_1\| \cdot \|v_2\|}
\]



- **Dot product (v₁·v₂)** → measures alignment between two vectors.  
- **Norm (‖v‖)** → measures the length (magnitude) of each vector.  
- **Division** → normalizes the result so only direction matters.  

**Range of values:**
- +1 → vectors point in exactly the same direction (perfect similarity).  
- 0 → vectors are orthogonal (no similarity).  
- -1 → vectors point in opposite directions (opposite meaning, rare in embeddings).



#### 3. Why Use Cosine Similarity for Embeddings
- Embedding vectors can have very large magnitudes depending on text length.  
- We care about **semantic meaning**, not raw magnitude.  
- Cosine similarity ignores magnitude and focuses only on **directional closeness**.  
- This makes it ideal for comparing sentences, documents, or queries in semantic search.



#### 4. Example in Practice
Suppose we embed three sentences:
- A: "Rohit Sharma hit a century in IPL"  
- B: "The batsman scored 100 runs in the tournament"  
- C: "The pitch was wet due to heavy rain"  

Results:
- Similarity(A, B) → high (close to 0.9) because both describe the same event.  
- Similarity(A, C) → low (close to 0.2) because the topics differ.  

This proves embeddings capture **meaning**, and cosine similarity quantifies how close those meanings are.



#### 5. Intuitive Analogy
Imagine each sentence as an arrow pointing in a direction in space:
- If two arrows point in the same direction → meanings are similar.  
- If arrows point in different directions → meanings diverge.  
Cosine similarity measures the **angle between arrows**. Smaller angle = higher similarity.

---

### Key Takeaway
Cosine similarity is the backbone of **semantic search** and **RAG pipelines**.  
It allows us to compare embeddings by direction, ensuring that sentences with similar meaning are recognized as close, even if they use different words.


In [19]:
# ── Similarity Demo — prove same meaning = similar vectors ──
import numpy as np

def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sentences = {
    "A": "Rohit Sharma hit a century in IPL",
    "B": "The batsman scored 100 runs in the tournament",   # similar meaning
    "C": "The pitch was wet due to heavy rain",              # different topic
}

vecs = {k: embeddings.embed_query(v) for k, v in sentences.items()}

print("A :", sentences["A"])
print("B :", sentences["B"])
print("C :", sentences["C"])
print()
print(f"Similarity A ↔ B (same meaning) : {cosine_similarity(vecs['A'], vecs['B']):.4f}")
print(f"Similarity A ↔ C (diff meaning) : {cosine_similarity(vecs['A'], vecs['C']):.4f}")
print("\n A↔B score MUCH higher than A↔C — this is how semantic search works!")

A : Rohit Sharma hit a century in IPL
B : The batsman scored 100 runs in the tournament
C : The pitch was wet due to heavy rain

Similarity A ↔ B (same meaning) : 0.4467
Similarity A ↔ C (diff meaning) : 0.1604

 A↔B score MUCH higher than A↔C — this is how semantic search works!



##  VECTOR STORE (FAISS)


## What is a Vector Store?

Now we have:
- Hundreds of chunks (from Text Splitter)
- Each chunk can become a 384-number vector (via Embeddings)

We need somewhere to **store** all these vectors AND be able to **search** them fast.

**A Vector Store is a special database that:**
1. Stores text chunks alongside their embedding vectors
2. Given a query vector → finds the **most similar** stored vectors instantly
3. Returns the matching text chunks as context for the LLM

---
### How Similarity Search Works

```
User asks: "Who scored the most runs?"
                │
                ▼
   Embed the query → query_vector [0.11, 0.84, ...]
                │
                ▼
   Compare query_vector to ALL stored vectors (cosine similarity)
                │
                ▼
   Return TOP-K most similar chunks  ← becomes the RAG context
```

---
### We Use: FAISS (Facebook AI Similarity Search)

| Property | Value |
|---|---|
| Cost | **Free & Open Source** |
| Runs where | Fully **local**, no cloud |
| Speed | Extremely fast |
| Persistable | Yes, save/load from disk |

**Other options (for reference):**

| Store | Where | Best For |
|---|---|---|
| **FAISS** | Local | Learning, prototypes |
| **Chroma** | Local | Local dev |
| **Pinecone** | Cloud | Production apps |
| **Weaviate** | Cloud/Local | Enterprise |

# Core Problems Solved by Vector Stores

---

## Problem 1 : Scaling Embedding Generation
Creating embeddings for small datasets (like 46 text chunks) is quick, often under a second on a CPU.  
However, scaling this to enterprise workloads is a major challenge.  
Imagine a law firm with **500,000 case files**, each split into **20 chunks**, resulting in **10 million total chunks**.  
Generating embeddings for all these on a single CPU would take **days**.  
Vector stores solve this through **batch processing**, **GPU acceleration**, and **offline pre-computation**, enabling large-scale embedding generation efficiently.

---

## Problem 2 : Inefficiency of Traditional Databases for Vector Storage
Conventional relational databases (e.g., MySQL, PostgreSQL) are optimized for structured data, text, numbers, and dates.  
While it’s technically possible to store vectors (like 384 floating-point values per row), querying them efficiently is not.  
For example, asking “find the 5 rows whose 384-number arrays are closest to this query vector” is not a native SQL operation.  
These databases lack built-in support for **vector distance calculations** (like cosine similarity or Euclidean distance).  
Hence, specialized **vector databases** are required, designed for **numerical vector storage and fast similarity retrieval**.

---

## Problem 3 : Slow Similarity Search at Scale
A brute-force similarity search compares the query vector against every stored vector individually.  
This works fine for small datasets but becomes impractical as data grows.

| Dataset Size | Comparisons per Query | Performance |
|---------------|----------------------|--------------|
| 46 chunks | 46 | Instant |
| 10,000 chunks | 10,000 | Acceptable |
| 10 million chunks | 10,000,000 | Too slow for real-time |

This linear growth (O(n) complexity) makes searches increasingly slow.  

Vector stores overcome this using **approximate nearest neighbor (ANN)** algorithms, **indexing structures** (like HNSW, IVF, PQ), and **optimized retrieval pipelines**.  
These techniques reduce search time from seconds to milliseconds, enabling **real-time semantic search** even at massive scale.

---

## Summary
Vector stores address three fundamental bottlenecks:
1. **Efficient embedding generation** for millions of chunks.  
2. **Purpose-built storage** for high-dimensional numerical vectors.  
3. **Fast similarity search** using advanced indexing and ANN algorithms.  

Together, they make large-scale **semantic retrieval** practical, accurate, and production-ready.


# Indexing: How Vector Stores Stay Fast

---

## 1. The Core Idea
Vector stores avoid brute-force comparisons by **organizing vectors into clusters**.  
Instead of checking every stored vector, the system first identifies the most relevant cluster and then searches only within that smaller group.  
This reduces the number of comparisons dramatically while keeping accuracy high.

---

## 2. Example : News Article Database
Imagine a database with **1 million articles**.  
The embedding model naturally groups articles by topic in vector space:
- Cricket articles in one region.  
- Politics articles in another.  
- Technology articles in another.  

We then create **1,000 clusters**, each represented by a **centroid** (the average vector of all items in that cluster).

---

## 3. Query Walkthrough
Query: *“Who won the T20 World Cup?”*

1. **Step 1 : Compare with Centroids**  
   The query vector is compared against 1,000 centroids.  
   → Only 1,000 comparisons.

2. **Step 2 : Identify Nearest Cluster**  
   The closest centroid corresponds to the **cricket cluster**.

3. **Step 3 : Search Within Cluster**  
   The query is compared against the ~1,000 articles inside that cluster.  
   → Another 1,000 comparisons.

**Total comparisons: ~2,000 instead of 1,000,000.**  
That’s a **500x speedup** with almost no loss in quality.

---

## 4. Approximate Nearest Neighbor (ANN)
This technique is called **ANN Search**.  
- “Approximate” means the system may miss the absolute best match occasionally.  
- In exchange, search becomes **dramatically faster**.  
- For RAG and semantic search applications, this tradeoff is almost always acceptable because results remain highly relevant.

---

## 5. Key Takeaway
Indexing ensures vector stores stay fast by:
- **Clustering vectors** into meaningful groups.  
- **Reducing comparisons** from millions to just a few thousand.  
- **Leveraging ANN algorithms** to balance speed and accuracy.  

This makes **real-time semantic search** practical even at massive scale.


# What is a Vector Store?

A **vector store** is a specialized system designed to manage embeddings (numerical representations of text, images, or other data).  
Its purpose is to make semantic search and retrieval efficient at scale.  
It does this by storing vectors, indexing them for fast lookup, and supporting updates as data changes.

---

## Core Functions
1. **Storage**  
   Saves each embedding vector along with metadata such as source file, chunk index, or custom tags.  
   This ensures traceability and context for retrieved results.

2. **Similarity Search**  
   Accepts a query vector and returns the **top‑K most similar chunks**, often with similarity scores.  
   This is the foundation of semantic search and RAG pipelines.

3. **Indexing**  
   Organizes vectors using clustering or tree structures so that searches run in **sub‑linear time**.  
   Instead of brute force, only relevant clusters are searched.

4. **CRUD Operations**  
   Supports adding new documents, updating changed ones, and deleting outdated entries.  
   This keeps the knowledge base current and reliable.

---

## Vector Store vs Vector Database

| Aspect | **Vector Store** | **Vector Database** |
|--------|------------------|----------------------|
| Primary Job | Store and search vectors | Vector store plus enterprise-grade features |
| User Authentication | Usually absent | Supported |
| Multi-user Access | Limited | Full support |
| Transactions & Rollback | No | Yes |
| Distributed Across Machines | No | Yes |
| Backup & Restore | Minimal | Full |
| Best Suited For | Learning, prototyping | Production systems |
| Examples | FAISS | Pinecone, Weaviate, Qdrant, Milvus |

---

## Key Insight
- **All vector databases are vector stores**, but not all vector stores qualify as full databases.  
- A simple vector store (like FAISS) is perfect for prototyping and experimentation.  
- Production systems require **vector databases** with enterprise features such as authentication, distributed scaling, and backup support.

In short, vector stores provide the **core mechanics of semantic search**, while vector databases extend those mechanics into **robust, production-ready infrastructure**.

# FAISS: What We Use for This Project

---

## What is FAISS?
FAISS stands for **Facebook AI Similarity Search**. It is a free, open-source library developed by Meta’s AI Research team.  

It is widely used for building vector stores in research and prototyping because it is lightweight, fast, and runs locally.

| Property     | Detail |
|--------------|--------|
| Cost         | Free, open source |
| Runs where   | Fully local, no account or internet needed at runtime |
| Speed        | Extremely fast, even on CPU for moderate data sizes |
| Persistence  | Can save the index to disk and reload later |
| Best for     | Learning, offline projects, rapid prototyping |

---

## How FAISS Processes Your IPL Chunks
1. **Input Data** → 46 IPL text chunks created by a Text Splitter.  
2. **Embedding** → Each chunk is embedded using a HuggingFace model, producing 46 vectors of length 384.  
3. **Index Creation** → `FAISS.from_documents(chunks, embeddings)` builds the FAISS index in memory.  
4. **Query** → User asks: *“How many wickets did Noor Ahmad take?”*  
5. **Query Embedding** → The question is embedded into a query vector of length 384.  
6. **Search** → FAISS compares the query vector against all 46 stored vectors.  
7. **Result** → Returns the top‑3 most similar chunks.  
8. **LLM Context** → Those chunks are passed to the LLM, which reads them and answers accurately.

This workflow shows how FAISS enables **semantic search** by combining embeddings with efficient vector comparison.

---

## Comparing Vector Store Options

| Store      | Hosted        | Paid | Best For |
|------------|---------------|------|----------|
| FAISS      | Local only    | Free | Learning, offline, fast prototyping |
| Chroma     | Local (SQLite)| Free | Local dev with persistence and basic DB features |
| Pinecone   | Cloud         | Paid | Managed production, no infrastructure to maintain |
| Weaviate   | Cloud or self-host | Free tier + Paid | Enterprise, supports images + text together |
| Qdrant     | Cloud or self-host | Free tier + Paid | High-performance production, open source |
| Milvus     | Self-hosted   | Free | Very large scale, open source enterprise |

---

## Key Insight
- **FAISS** is ideal for **offline experiments, classroom demos, and rapid prototyping**.  
- For production systems, cloud-hosted vector databases like **Pinecone, Weaviate, Qdrant, or Milvus** provide enterprise features such as authentication, scaling, and backups.  
- In short: FAISS is the **learning tool**, while vector databases are the **production engines**.

In [20]:
from langchain_community.vectorstores import FAISS

# Embeds every chunk and stores (text, vector) pairs in the FAISS index
print("Building vector store — embedding all chunks (30-60 sec)...")

vectorstore = FAISS.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

print(f"Vector Store ready!")
print(f"   Vectors stored : {vectorstore.index.ntotal}")

Building vector store — embedding all chunks (30-60 sec)...
Vector Store ready!
   Vectors stored : 46


In [21]:
# ── Similarity Search — raw demo ──
query = "Who took the most wickets?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: '{query}'")
print(f"Top {len(results)} matching chunks:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source  : {doc.metadata.get('source', 'unknown')}")
    print(f"Content : {doc.page_content[:200]}")
    print()

Query: 'Who took the most wickets?'
Top 3 matching chunks:

--- Result 1 ---
Source  : C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt
Content : Fall of Wickets (DC):
1st wicket: 29 runs (Nissanka, over 3.6)
2nd wicket: 36 runs (KL Rahul, over 5.1)
3rd wicket: 52 runs (Karun Nair, over 7.6)
4th wicket: 61 runs (Nitish Rana, over 9.3)
5th wicke

--- Result 2 ---
Source  : C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt
Content : Fall of Wickets (CSK):
1st wicket: 24 runs (Ruturaj Gaikwad, over 3.5)
2nd wicket: 45 runs (Urvil Patel, over 6.3)

--- Result 3 ---
Source  : C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt
Content : === CSK BOWLING (vs DC) ===
Akeal Hosein: 4 overs, 0 maiden, 19 runs, 1 wicket, Econ: 4.75
Mukesh Choudhary: 4 overs, 0 maiden, 31 runs, 1 wicket, Econ: 7.75
Anshul Kamboj: 4 overs, 0 maiden, 49 runs,



In [22]:
# ── With Score — shows how similar each result is ──
results_scored = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: '{query}' — with scores (lower = more similar in FAISS L2)\n")
for doc, score in results_scored:
    print(f"Score   : {score:.4f}")
    print(f"Content : {doc.page_content[:150]}")
    print()

Query: 'Who took the most wickets?' — with scores (lower = more similar in FAISS L2)

Score   : 0.7286
Content : Fall of Wickets (DC):
1st wicket: 29 runs (Nissanka, over 3.6)
2nd wicket: 36 runs (KL Rahul, over 5.1)
3rd wicket: 52 runs (Karun Nair, over 7.6)
4th

Score   : 0.7857
Content : Fall of Wickets (CSK):
1st wicket: 24 runs (Ruturaj Gaikwad, over 3.5)
2nd wicket: 45 runs (Urvil Patel, over 6.3)

Score   : 0.7953
Content : === CSK BOWLING (vs DC) ===
Akeal Hosein: 4 overs, 0 maiden, 19 runs, 1 wicket, Econ: 4.75
Mukesh Choudhary: 4 overs, 0 maiden, 31 runs, 1 wicket, Eco



In [23]:
# ── Save & Load FAISS index (production pattern) ──
FAISS_PATH = r"C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\faiss_index"

# Save
vectorstore.save_local(FAISS_PATH)
print(f"Saved to: {FAISS_PATH}")

# Load (next time no need to re-embed everything)
vectorstore_loaded = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True   # Required in LangChain 2026
)
print(f"Loaded from disk — vectors: {vectorstore_loaded.index.ntotal}")

Saved to: C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\faiss_index
Loaded from disk — vectors: 46


# RETRIEVER

---

## Theory: What is a Retriever?

A **Retriever** is a lightweight interface that sits on top of a **Vector Store**.  
Its main purpose is to make the vector store usable inside LangChain Expression Language (LCEL) pipelines, because LCEL requires components to be **Runnables**.  
Think of the retriever as a translator: it takes a query, interacts with the vector store, and returns results in a format that fits into the chain.



## Vector Store vs Retriever

| Aspect | Vector Store | Retriever |
|--------|--------------|-----------|
| What it is | The database holding embeddings | Interface to the database |
| How to call | `.similarity_search(query)` | `.invoke(query)` |
| Works in LCEL `|` chain | No | Yes |
| Returns | List of Documents | List of Documents |

**Example:**  
- If you only want to test similarity search, you can call the vector store directly.  
- If you want to plug the search into a pipeline (e.g., query → retriever → LLM), you must use the retriever.

---

## Retriever Search Types

| Type | How It Works | Best When |
|------|--------------|-----------|
| `similarity` | Uses cosine similarity to rank vectors | Default choice; works well in most cases |
| `mmr` | Max Marginal Relevance, balances relevance and diversity | When results feel repetitive and you want variety |
| `similarity_score_threshold` | Filters out results below a similarity score | When precision matters more than quantity |

**Example Scenarios:**
- `similarity`: A student asks “Explain Kohli’s batting average.” → Retriever returns the most relevant chunks about Kohli.  
- `mmr`: A researcher asks “AI applications in healthcare.” → Retriever returns diverse chunks covering diagnostics, drug discovery, patient monitoring.  
- `similarity_score_threshold`: A lawyer queries “Case law on intellectual property.” → Retriever ensures only highly relevant case chunks are returned, ignoring weak matches.

---

## Key Takeaway
The retriever is not a replacement for the vector store, it is the **bridge** that makes the vector store usable in pipelines.  
By choosing the right search type, you can control whether results are purely relevant, diverse, or filtered for quality.


In [24]:
# ── Basic Retriever ──
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # Return top 4 most relevant chunks
)

# Test — retriever is a Runnable so use .invoke()
retrieved = retriever.invoke("Which team is leading IPL?")

print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved):
    print(f"[{i+1}] {doc.page_content[:200]}")
    print()

Retrieved 4 chunks:

[1] IPL 2026 - Match 48 (Night Game)
Venue: Delhi, May 05, 2026
Indian Premier League

=== RESULT ===
Chennai Super Kings beat Delhi Capitals by 8 wickets (with 15 balls remaining)
Delhi Capitals scored 1

[2] match: IPL2026_M48
innings: 1st innings
team_bowling: Chennai Super Kings
bowler: Noor Ahmad
overs: 3
maidens: 0
runs: 22
wickets: 2
economy: 7.33
dot_balls: 6
wides: 0
no_balls: 0

[3] match: IPL2026_M48
innings: 1st innings
team_bowling: Chennai Super Kings
bowler: Jamie Overton
overs: 1
maidens: 0
runs: 5
wickets: 1
economy: 5.00
dot_balls: 1
wides: 0
no_balls: 0

[4] match: IPL2026_M48
innings: 2nd innings
team_bowling: Delhi Capitals
bowler: Axar Patel
overs: 4
maidens: 0
runs: 25
wickets: 1
economy: 6.25
dot_balls: 12
wides: 1
no_balls: 0



In [32]:
retrieved

[Document(id='a2f9d8c5-a604-4e37-8606-174b8cbf2eb5', metadata={'source': 'C:\\Users\\admin\\Desktop\\New_GenAI\\GenAI\\Langchain\\Doc_loader\\match_summary.txt', 'start_index': 0}, page_content='IPL 2026 - Match 48 (Night Game)\nVenue: Delhi, May 05, 2026\nIndian Premier League\n\n=== RESULT ===\nChennai Super Kings beat Delhi Capitals by 8 wickets (with 15 balls remaining)\nDelhi Capitals scored 155/7 in 20 overs.\nChennai Super Kings chased it down in 17.3 overs, finishing at 159/2.\n\n=== PLAYER OF THE MATCH ===\nSanju Samson (CSK) - 87* off 52 balls (7 fours, 6 sixes, SR: 167.30)\nCricinfo MVP Points: 105.96'),
 Document(id='1cfd812a-acce-4855-b28f-ecca857e13ff', metadata={'source': 'C:\\Users\\admin\\Desktop\\New_GenAI\\GenAI\\Langchain\\Doc_loader\\bowling_figures.csv', 'row': 3, 'start_index': 0}, page_content='match: IPL2026_M48\ninnings: 1st innings\nteam_bowling: Chennai Super Kings\nbowler: Noor Ahmad\novers: 3\nmaidens: 0\nruns: 22\nwickets: 2\neconomy: 7.33\ndot_balls: 6\n

In [25]:
# ── MMR Retriever — avoids near-duplicate chunks ──
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 10}   # fetch 10, keep most diverse 4
)
mmr_docs = mmr_retriever.invoke("batting performance")
print(f"MMR → {len(mmr_docs)} diverse chunks retrieved")

MMR → 4 diverse chunks retrieved


In [26]:
# ── format_docs — converts list of Documents to one context string ──
# This is called INSIDE the RAG chain to prepare the {context} for the prompt
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# Test it
test_context = format_docs(retriever.invoke("best bowler"))
print(test_context[:600])

[Source: C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\match_summary.txt]
=== CSK BOWLING (vs DC) ===
Akeal Hosein: 4 overs, 0 maiden, 19 runs, 1 wicket, Econ: 4.75
Mukesh Choudhary: 4 overs, 0 maiden, 31 runs, 1 wicket, Econ: 7.75
Anshul Kamboj: 4 overs, 0 maiden, 49 runs, 0 wickets, Econ: 12.25
Noor Ahmad: 3 overs, 0 maiden, 22 runs, 2 wickets, Econ: 7.33
Gurjapneet Singh: 4 overs, 0 maiden, 29 runs, 1 wicket, Econ: 7.25
Jamie Overton: 1 over, 0 maiden, 5 runs, 1 wicket, Econ: 5.00

=== POST MATCH QUOTES ===

[Source: C:\Users\admin\Desktop\New_GenAI\GenAI\Langchain\Doc_loader\


---
# ══════════════════════════════════════════════════
# FULL RAG CHAIN (LCEL)
# ══════════════════════════════════════════════════

## Theory : What is RAG?

**RAG = Retrieval-Augmented Generation**

A plain LLM like LLaMA only knows what it was trained on (data up to a cutoff date).  
It has **zero knowledge** of your CSV files, your match summary, or the live IPL table.

**RAG bridges this gap:**
1. Takes the user's question
2. **Retrieves** the most relevant chunks from YOUR data
3. Injects those chunks as **context** into the prompt
4. Lets the LLM **generate** an answer grounded in YOUR data

```
User: "Who scored the most runs?"
       │
       ├──► Retriever ──► finds "Rohit: 85, Kohli: 72..." from batting_scorecard.csv
       │                               │
       └───────────────────────────────▼
                     Prompt:  SYSTEM = "Use this context: Rohit: 85, Kohli: 72..."
                              HUMAN  = "Who scored the most runs?"
                                       │
                                       ▼
                                 Groq LLaMA 3.3
                                       │
                                       ▼
                          "Rohit Sharma scored the most with 85 runs."
```

---
## 🔑 Why LCEL Instead of Old Chains?

Old LangChain (deprecated, DO NOT USE):
```python
# Old — deprecated in 2025
chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
```

New LCEL (2026: use this):
```python
# New — transparent, composable, production-ready
chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
         | rag_prompt | llm | StrOutputParser())
```

LCEL benefits:
- Every step is visible and debuggable
- Easily add/remove steps
- Streaming works out of the box
- Parallel execution supported natively

In [27]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# ════════════════════════════════════════════════════
# THE RAG CHAIN — How it works step by step:
#
# RunnableParallel runs TWO branches simultaneously:
#   Branch 1 "context"  : question → retriever → format_docs → string
#   Branch 2 "question" : question passes through unchanged
#
# Both outputs feed into rag_prompt as {context} and {question}
# Then: rag_prompt → llm → StrOutputParser
# ════════════════════════════════════════════════════

rag_chain = (
    RunnableParallel({
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    })
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain ready — pipeline: Retriever → Prompt → LLM → Parser")

RAG Chain ready — pipeline: Retriever → Prompt → LLM → Parser


In [28]:
# ── Test 1 — Batting Question ──
q = "Who scored the highest runs in the batting scorecard?"
print(f"Q: {q}")
print(f"A: {rag_chain.invoke(q)}")

Q: Who scored the highest runs in the batting scorecard?
A: Sameer Rizvi scored the highest runs with 40* off 24 balls.


In [29]:
# ── Test 1 — Batting Question ──
q = "performance of noor ahmed"
print(f"Q: {q}")
print(f"A: {rag_chain.invoke(q)}")

Q: performance of noor ahmed
A: Noor Ahmad's performance: 
- Overs: 3
- Maidens: 0
- Runs: 22
- Wickets: 2
- Economy: 7.33 

He also got 2 batters from Delhi Capitals out, namely Nitish Rana and Karun Nair.


In [30]:
print("🏏 Cricket RAG Assistant — Ask anything about the match data!")
print("Type 'quit' to exit\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ["quit", "exit", "q"]:
        print("Bye!")
        break
    if not question:
        continue
    answer = rag_chain.invoke(question)
    print(f"\n🤖 Bot: {answer}\n")
    print("-" * 50)

🏏 Cricket RAG Assistant — Ask anything about the match data!
Type 'quit' to exit

Bye!


# Cosine Similarity vs Centroid-Based Vector Search vs Retrieval

---

## 1. Cosine Similarity
- **What it is:** A mathematical measure of how close two vectors are in direction.  
- **How it works:** Given a query vector and a stored vector, cosine similarity calculates the angle between them. Smaller angle → higher similarity.  
- **Use case:** Directly compares one query against one document chunk.  
- **Limitation:** If you have millions of vectors, computing cosine similarity against all of them becomes too slow.

**Example:**  
Query: "Rohit Sharma scored a century"  
Stored chunk: "Batsman made 100 runs in IPL"  
Cosine similarity ≈ 0.9 → very similar.

---

## 2. Centroid-Based Vector Search (Indexing / ANN)
- **What it is:** A way to make similarity search faster by grouping vectors into clusters.  
- **How it works:**  
  1. Vectors are organized into clusters, each with a centroid (average vector).  
  2. The query is first compared to centroids (fast).  
  3. Only the nearest cluster is searched in detail using cosine similarity.  
- **Use case:** Large-scale datasets (millions of vectors).  
- **Tradeoff:** It’s “approximate” — you may miss the absolute best match, but you gain huge speed improvements.

**Example:**  
Database: 1 million news articles clustered into 1,000 groups.  
Query: "Who won the T20 World Cup?"  
Step 1: Compare query with 1,000 centroids → nearest is cricket cluster.  
Step 2: Compare query only with ~1,000 cricket articles.  
Result: ~2,000 comparisons instead of 1,000,000.

---

## 3. Retriever
- **What it is:** A wrapper/interface around the vector store that makes it usable in pipelines (like RAG).  
- **How it works:**  
  - Accepts a query.  
  - Calls the vector store (which internally uses cosine similarity + indexing).  
  - Returns the top‑K most relevant chunks.  
- **Use case:** Needed when integrating with LLMs, because retrievers are “runnables” that can plug into chains.  
- **Benefit:** Abstracts away the complexity — you just call `.invoke(query)` and get results.

**Example:**  
User asks: "How many wickets did Noor Ahmad take?"  
Retriever:  
1. Embeds the query.  
2. Uses vector store (FAISS) to run similarity search.  
3. Returns top‑3 relevant IPL chunks.  
4. Passes them to the LLM for answering.

---

## Key Differences
- **Cosine similarity** → the actual math for comparing two vectors.  
- **Centroid-based vector search (ANN)** → optimization technique to avoid brute-force cosine similarity across all vectors.  
- **Retriever** → the interface that connects queries to the vector store, making it usable in RAG pipelines.

---

## Takeaway
Think of it like this:  
- **Cosine similarity** is the *ruler* that measures closeness.  
- **Centroid/ANN indexing** is the *shortcut* that avoids measuring everything.  
- **Retriever** is the *messenger* that takes your query, uses the shortcut + ruler, and delivers the most relevant chunks to the LLM.


# Cosine Similarity vs Vector Store Indexing vs Retriever

---

## 1. Cosine Similarity
- **Definition:** A mathematical formula that measures how close two vectors are in direction.  
- **Where it’s used:**  
  - Inside the **vector store** when comparing a query vector against stored vectors.  
  - Inside the **retriever**, because the retriever calls the vector store and relies on cosine similarity to rank results.  
- **Key Point:** Cosine similarity is the *metric* used to judge closeness between embeddings.

**Example:**  
Query vector for “Rohit Sharma century” compared with stored vector for “Batsman scored 100 runs” → cosine similarity ≈ 0.9 (high match).

---

## 2. Vector Store Indexing (Centroids / ANN)
- **Definition:** An optimization technique to avoid brute-force cosine similarity across millions of vectors.  
- **How it works:**  
  - Vectors are grouped into clusters with centroids.  
  - Query is first compared to centroids (fast).  
  - Only the nearest cluster is searched in detail using cosine similarity.  
- **Key Point:** Indexing reduces the number of cosine similarity calculations needed, making search scalable.

**Example:**  
Instead of comparing against 1,000,000 articles, the query first checks 1,000 centroids, then only searches ~1,000 articles in the nearest cluster.  
Total ≈ 2,000 comparisons instead of 1,000,000.

---

## 3. Retriever
- **Definition:** A wrapper around the vector store that makes it usable in pipelines (like RAG).  
- **How it works:**  
  - Accepts a query.  
  - Calls the vector store (which internally uses cosine similarity + indexing).  
  - Returns the top‑K most relevant chunks.  
- **Key Point:** The retriever doesn’t invent a new metric — it simply exposes vector store search (which uses cosine similarity) in a pipeline-friendly way.

**Example:**  
User asks: “How many wickets did Noor Ahmad take?”  
Retriever → embeds query → vector store runs cosine similarity search → returns top‑3 relevant IPL chunks → LLM answers.

---

## Final Clarification
- **Cosine similarity** is the *mathematical measure* used everywhere to compare vectors.  
- **Indexing/ANN** is the *optimization layer* that reduces how many cosine similarity checks are needed.  
- **Retriever** is the *interface* that plugs vector store search (with cosine similarity + indexing) into RAG pipelines.

So yes — cosine similarity is used in both **vector store search** and **retriever**, but indexing decides *how many* cosine similarity calculations are performed.
